# 04. Fine-Tuning y Comparativa de Fashion-CLIP vs Adaptador Estético

Este notebook implementa el pipeline completo de Fine-Tuning de compatibilidad estética.

### Estrategia:
1. **Backbone Congelado:** Se mantiene el extractor visual de `patrickjohncyh/fashion-clip` intacto para preservar el conocimiento previo de moda sin catastrophic forgetting.
2. **Adaptador de Compatibilidad D=128:** Se entrena una cabeza densa ligera con Contrastive Loss (InfoNCE) sobre los 12,000 pares de prendas reales (`sample_data/clip_pairs_*.jsonl`) y clasificación sobre 12 estéticas ampliadas.
3. **Calibración de Temperatura:** Se ajusta el parámetro T para obtener probabilidades calibradas.
4. **Comparativa Lado a Lado:** Se mide el Baseline Zero-Shot vs el Adaptador en F1-Macro y FITB (Fill-In-The-Blank, meta >= 70%), registrando ambos experimentos en DagsHub / MLflow.
5. **Exportación a Google Drive:** Guarda el archivo `fashion_clip_adapter.pt` directamente en Google Drive (`StyleMe/models/`) para descargarlo y utilizarlo en el servidor local.


In [ ]:
import platform
import sys
from pathlib import Path

# Detección automática de entorno (Colab vs Local)
IS_COLAB = Path("/content").exists()

if IS_COLAB:
    REPO_ROOT = Path("/content/MACHINE-BULLS") if Path("/content/MACHINE-BULLS").exists() else Path("/content")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("[OK] Google Drive montado exitosamente en /content/drive")
    except Exception as exc:
        print(f"[WARN] No se pudo montar Google Drive: {exc}")
    print("[OK] Entorno Google Colab detectado")
else:
    REPO_ROOT = Path.cwd().resolve()
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if (parent / "notebooks" / "04_clip_fine_tuning.ipynb").exists() or (parent / "sample_data").exists():
            REPO_ROOT = parent
            break
    print(f"[OK] Entorno Local detectado | Repo: {REPO_ROOT}")

print(f"Python: {sys.version.split()[0]} | Plataforma: {platform.platform()}")


In [ ]:
# Instalación de dependencias en Google Colab
if IS_COLAB:
    !pip install -q transformers dagshub mlflow pyyaml pillow
    print("[OK] Dependencias instaladas en Google Colab")


In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")
load_dotenv(REPO_ROOT / "backend" / ".env")

# Configuración de secretos para DagsHub y MLflow
token = os.environ.get("DAGSHUB_USER_TOKEN") or os.environ.get("DAGSHUB_TOKEN")
if token:
    os.environ["DAGSHUB_TOKEN"] = token

import dagshub
import mlflow

try:
    dagshub.init(repo_owner="jesusrodrigov", repo_name="machine-bulls", mlflow=True)
    print("[OK] Conectado exitosamente a DagsHub MLflow")
except Exception as ex:
    print(f"[INFO] MLflow local: {ex}")


In [ ]:
# Asegurar que el modulo src de backend este en el path
backend_src = REPO_ROOT / "backend" / "src"
if str(backend_src) not in sys.path:
    sys.path.insert(0, str(backend_src))

import torch
from ml.augmentation import verificar_splits_anti_fuga, get_image_transforms
from ml.models.adapter import AestheticProjectionHead
from ml.models.prompts import ESTETICAS_AMPLIADAS
from ml.training.train import train, load_config
from ml.validation.evaluate_fitb import evaluate_fitb
from ml.validation.temperature_scaling import ModelWithTemperature
from ml.validation.run_comparison import run_baseline_evaluation, run_adapter_evaluation, print_comparison_table

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[OK] Módulos de StyleSync importados correctamente. Dispositivo activo: {device}")
print(f"[INFO] Catálogo ampliado de estéticas ({len(ESTETICAS_AMPLIADAS)}): {list(ESTETICAS_AMPLIADAS)}")


In [ ]:
# Ejecutar entrenamiento del adaptador (D=512 -> D=128) sobre los pares reales
config_path = REPO_ROOT / "backend" / "src" / "ml" / "config.yaml"
cfg = load_config(config_path) if config_path.exists() else {
    "model": "patrickjohncyh/fashion-clip",
    "dataset": {"name": "sample_data_clip_pairs"},
    "training": {"epochs": 5 if IS_COLAB else 1, "batch_size": 32 if IS_COLAB else 8, "lr": 0.001},
    "seed": 42
}

print(">>> Iniciando entrenamiento del Adaptador Estético...")
train_results = train(cfg, fast_dev_run=False, use_mlflow=True, run_name="colab-fashion-clip-adapter")
print("[OK] Entrenamiento completado:", train_results)


In [ ]:
# Guardar y respaldar checkpoint en Google Drive para descarga local
import shutil

local_ckpt = Path(train_results["checkpoint_path"])
if IS_COLAB and Path("/content/drive/MyDrive").exists():
    drive_folder = Path("/content/drive/MyDrive/StyleMe/models")
    drive_folder.mkdir(parents=True, exist_ok=True)
    drive_dest = drive_folder / "fashion_clip_adapter.pt"
    shutil.copy(local_ckpt, drive_dest)
    print(f"[EXITO] Checkpoint copiado a Google Drive en: {drive_dest}")
    print(">>> Descarga fashion_clip_adapter.pt de Google Drive y colócalo en 'models/fashion_clip_adapter.pt' de tu proyecto local.")
else:
    print(f"[INFO] Checkpoint local disponible en: {local_ckpt}")


In [ ]:
# Ejecutar comparativa formal completa (Baseline vs Adaptador)
b_metrics = run_baseline_evaluation()
a_metrics = run_adapter_evaluation()
print_comparison_table(b_metrics, a_metrics)
